In [1]:
import sys
import pandas as pd
import numpy as np

from pathlib import Path

project_root = Path.cwd().parents[1]

sys.path.insert(0, str(project_root / "lib"))

import BTMS_model

excel_path = project_root / "data" / "MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx"


N_cell = 320
dt_sample = 1.0

delta_t = 60

df = pd.read_excel(excel_path, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

q_cell = pd.to_numeric(df["Qbat(W)"], errors="coerce")
q_pack = N_cell * q_cell
q_pack = q_pack.dropna().reset_index(drop=True)

time_s = np.arange(1, len(q_pack) + 1) * dt_sample

Q_peak = q_pack.max()
idx_peak = q_pack.idxmax()
t_peak = time_s[idx_peak]

Q_08_peak = 0.8 * Q_peak

window_points = int(delta_t / dt_sample)

q_avg_series = q_pack.rolling(
    window=window_points,
    min_periods=window_points
).mean()

Q_avg_dt = q_avg_series.max()
idx_end = q_avg_series.idxmax()
idx_start = idx_end - window_points + 1

Q_design = max(Q_avg_dt, Q_08_peak)

heat_load_summary = pd.DataFrame([
    {
        "item": "Q_peak",
        "value_W": Q_peak,
        "time_s": t_peak
    },
    {
        "item": f"Q_avg_{delta_t}s",
        "value_W": Q_avg_dt,
        "time_s": f"{time_s[idx_start]:.0f}-{time_s[idx_end]:.0f}"
    },
    {
        "item": "0.8Q_peak",
        "value_W": Q_08_peak,
        "time_s": "-"
    },
    {
        "item": "Q_design",
        "value_W": Q_design,
        "time_s": "-"
    }
])

heat_load_summary["value_W"] = heat_load_summary["value_W"].round(3)

display(heat_load_summary)

print(f"delta_t = {delta_t} s")
print(f"Q_peak = {Q_peak:.3f} W")
print(f"Q_avg_dt = {Q_avg_dt:.3f} W")
print(f"0.8Q_peak = {Q_08_peak:.3f} W")
print(f"Q_design = {Q_design:.3f} W")


,item,value_W,time_s
0,Q_peak,1329.005,1920.0
1,Q_avg_60s,1276.329,1861-1920
2,0.8Q_peak,1063.204,-
3,Q_design,1276.329,-


delta_t = 60 s
Q_peak = 1329.005 W
Q_avg_dt = 1276.329 W
0.8Q_peak = 1063.204 W
Q_design = 1276.329 W


In [2]:
# ============================================================
# Cell 2 - Step 2: Calculate the required coolant mass flow rate
# ============================================================

cp_f = 4180.0              # J/(kg·K), water

# 可修改参数
DeltaT_f_allow = 5.0       # K, allowable coolant temperature rise

# 冷却液总质量流量
m_dot_f = Q_design / (cp_f * DeltaT_f_allow)

flow_rate_summary = pd.DataFrame([
    {
        "Q_design_W": Q_design,
        "cp_f_J_kgK": cp_f,
        "DeltaT_f_allow_K": DeltaT_f_allow,
        "m_dot_f_kg_s": m_dot_f
    }
])

flow_rate_summary = flow_rate_summary.round(4)

display(flow_rate_summary)

print("Step 2 results")
print("-" * 60)
print(f"Q_design = {Q_design:.3f} W")
print(f"DeltaT_f_allow = {DeltaT_f_allow:.1f} K")
print(f"cp_f = {cp_f:.1f} J/(kg·K)")
print(f"m_dot_f = {m_dot_f:.5f} kg/s")

,Q_design_W,cp_f_J_kgK,DeltaT_f_allow_K,m_dot_f_kg_s
0,1276.3287,4180.0,5.0,0.0611


Step 2 results
------------------------------------------------------------
Q_design = 1276.329 W
DeltaT_f_allow = 5.0 K
cp_f = 4180.0 J/(kg·K)
m_dot_f = 0.06107 kg/s


In [3]:
T_b_lim = 45.0          # °C, battery temperature limit
T_f_in = 25.0           # °C, coolant inlet temperature

# 冷却液总热容量率
C_f = m_dot_f * cp_f

# 整块冷板的最大可能换热量
Q_max = C_f * (T_b_lim - T_f_in)

# 整块冷板所需换热有效度
epsilon = Q_design / Q_max

design_condition_summary = pd.DataFrame([
    {
        "Q_design_W": Q_design,
        "T_b_lim_C": T_b_lim,
        "T_f_in_C": T_f_in,
        "C_f_W_K": C_f,
        "Q_max_W": Q_max,
        "epsilon": epsilon
    }
])

display(design_condition_summary.round(4))

print("Step 3 results")
print("-" * 60)
print(f"Q_design = {Q_design:.3f} W")
print(f"T_b_lim = {T_b_lim:.1f} °C")
print(f"T_f_in = {T_f_in:.1f} °C")
print(f"C_f = {C_f:.3f} W/K")
print(f"Q_max = {Q_max:.3f} W")
print(f"epsilon = {epsilon:.6f}")

,Q_design_W,T_b_lim_C,T_f_in_C,C_f_W_K,Q_max_W,epsilon
0,1276.3287,45.0,25.0,255.2657,5105.315,0.25


Step 3 results
------------------------------------------------------------
Q_design = 1276.329 W
T_b_lim = 45.0 °C
T_f_in = 25.0 °C
C_f = 255.266 W/K
Q_max = 5105.315 W
epsilon = 0.250000


In [4]:
# ============================================================
# Cell 4 - Step 4: Calculate the required total UA
# ============================================================

# 在电池侧温度近似保持为 T_b_lim 的恒温壁面假设下
NTU = -np.log(1.0 - epsilon)

UA_design = C_f * NTU

UA_design_summary = pd.DataFrame([
    {
        "Q_design_W": Q_design,
        "Q_max_W": Q_max,
        "epsilon": epsilon,
        "NTU": NTU,
        "C_f_W_K": C_f,
        "UA_design_W_K": UA_design
    }
])

display(UA_design_summary.round(4))

print("Step 4 results")
print("-" * 60)
print(f"Q_design = {Q_design:.3f} W")
print(f"Q_max = {Q_max:.3f} W")
print(f"epsilon = {epsilon:.6f}")
print(f"NTU = {NTU:.6f}")
print(f"C_f = {C_f:.3f} W/K")
print(f"UA_design = {UA_design:.3f} W/K")

,Q_design_W,Q_max_W,epsilon,NTU,C_f_W_K,UA_design_W_K
0,1276.3287,5105.315,0.25,0.2877,255.2657,73.4354


Step 4 results
------------------------------------------------------------
Q_design = 1276.329 W
Q_max = 5105.315 W
epsilon = 0.250000
NTU = 0.287682
C_f = 255.266 W/K
UA_design = 73.435 W/K


In [5]:
# ============================================================
# Cell 5 - Rectangular Parallel-Channel Sizing
# ============================================================

rho_f = 997.0              # Coolant density, kg/m3
mu_f = 0.00089             # Coolant dynamic viscosity, Pa·s
k_f = 0.606                # Coolant thermal conductivity, W/(m·K)
k_plate = 200.0            # Cold-plate thermal conductivity, W/(m·K)

# ------------------------------------------------------------
# Design assumptions
# ------------------------------------------------------------

N_ch = 20                  # Number of parallel channels
W_ch = 0.008               # Channel width, m, 8 mm
S_ch = 0.01777            # Clear spacing between channels, m, 12.18 mm

# Cold-plate width:
# 20 channels + 19 internal gaps + 2 edge gaps
W_plate = N_ch * W_ch + (N_ch + 1) * S_ch

delta_plate_coolant = 0.002  # Distance from heated surface to coolant, m
L_min = 0.379                # Minimum channel length, m

# Coolant mass flow rate through each parallel channel
m_dot_ch = m_dot_f / N_ch

# ------------------------------------------------------------
# Scan candidate channel heights
# ------------------------------------------------------------

H_ch_values_mm = np.round(np.arange(1.0, 8.0 + 0.05, 0.05), 2)

scan_results = []

for H_ch_mm in H_ch_values_mm:

    H_ch_i = H_ch_mm / 1000.0

    # Cross-sectional area of one rectangular channel
    A_ch_i = W_ch * H_ch_i

    # Wetted perimeter of one rectangular channel
    P_wetted_i = 2.0 * (W_ch + H_ch_i)

    # Hydraulic diameter
    D_h_i = 4.0 * A_ch_i / P_wetted_i

    # Coolant velocity in one channel
    u_ch_i = m_dot_ch / (rho_f * A_ch_i)

    # Reynolds number
    Re_i = rho_f * u_ch_i * D_h_i / mu_f

    # Retain only laminar-flow candidates
    if Re_i >= 2300.0:
        continue

    # Nusselt number for a rectangular channel
    Nu_i = BTMS_model.liquid_nusselt_number_rect_channel(W_ch, H_ch_i)

    # Coolant-side convective heat-transfer coefficient
    h_f_i = Nu_i * k_f / D_h_i

    # Overall heat-transfer coefficient
    U_i = 1.0 / (1.0 / h_f_i + delta_plate_coolant / k_plate)

    # Total effective heat-transfer area required to satisfy the design UA
    A_required_i = UA_design / U_i

    # Three heated channel surfaces: channel base and two side walls
    P_HT_i = W_ch + 2.0 * H_ch_i

    # Required channel length for all parallel channels
    L_ch_i = A_required_i / (N_ch * P_HT_i)

    scan_results.append({
        "H_ch_mm": H_ch_mm,
        "D_h_mm": D_h_i * 1000.0,
        "u_ch_m_s": u_ch_i,
        "Re": Re_i,
        "Nu": Nu_i,
        "h_f_W_m2K": h_f_i,
        "U_W_m2K": U_i,
        "A_required_m2": A_required_i,
        "P_HT_mm": P_HT_i * 1000.0,
        "L_ch_mm": L_ch_i * 1000.0
    })


# ============================================================
# Retain all feasible designs above 379 mm
# ============================================================

scan_table = pd.DataFrame(scan_results)

if scan_table.empty:
    raise ValueError("No channel-height candidate satisfies Re < 2300.")

# Retain all designs with a channel length strictly greater than 379 mm
feasible_scan_table = scan_table.loc[scan_table["L_ch_mm"] > L_min * 1000.0].copy()

if feasible_scan_table.empty:
    raise ValueError(f"No candidate with L_ch > {L_min * 1000.0:.0f} mm ""was found within the current channel-height range.")

# Sort feasible designs by channel length without selecting one design
feasible_scan_table = feasible_scan_table.sort_values(
    by=["L_ch_mm", "H_ch_mm"],
    ascending=[True, True]
).reset_index(drop=True)

# Calculate the length margin above 379 mm
feasible_scan_table["Length_above_379_mm"] = (feasible_scan_table["L_ch_mm"] - L_min * 1000.0)


# ============================================================
# Design assumptions summary
# ============================================================

design_assumptions = pd.DataFrame([{
    "Channel_number": N_ch,
    "Channel_width_mm": W_ch * 1000.0,
    "Channel_spacing_mm": S_ch * 1000.0,
    "Edge_spacing_each_side_mm": S_ch * 1000.0,
    "Cold_plate_width_mm": W_plate * 1000.0,
    "Plate_to_coolant_distance_mm": delta_plate_coolant * 1000.0,
    "Minimum_channel_length_mm": L_min * 1000.0,
    "Total_mass_flow_rate_kg_s": m_dot_f,
    "Mass_flow_rate_per_channel_kg_s": m_dot_ch,
    "Number_of_feasible_designs": len(feasible_scan_table)
}])

display(design_assumptions.round(6))


# ============================================================
# Display all feasible designs
# ============================================================

display(
    feasible_scan_table[[
        "H_ch_mm",
        "D_h_mm",
        "u_ch_m_s",
        "Re",
        "Nu",
        "h_f_W_m2K",
        "U_W_m2K",
        "A_required_m2",
        "P_HT_mm",
        "L_ch_mm",
        "Length_above_379_mm"
    ]].round(6)
)


# ============================================================
# Print scan information
# ============================================================

print("Rectangular parallel-channel design scan")
print("-" * 65)
print(f"Channel number                 : {N_ch}")
print(f"Channel width                  : {W_ch * 1000.0:.2f} mm")
print(f"Channel spacing                : {S_ch * 1000.0:.2f} mm")
print(f"Edge spacing on each side      : {S_ch * 1000.0:.2f} mm")
print(f"Cold-plate width               : {W_plate * 1000.0:.2f} mm")
print(f"Plate-to-coolant distance      : {delta_plate_coolant * 1000.0:.2f} mm")
print(f"Minimum channel length         : > {L_min * 1000.0:.2f} mm")

,Channel_number,Channel_width_mm,Channel_spacing_mm,Edge_spacing_each_side_mm,Cold_plate_width_mm,Plate_to_coolant_distance_mm,Minimum_channel_length_mm,Total_mass_flow_rate_kg_s,Mass_flow_rate_per_channel_kg_s,Number_of_feasible_designs
0,20,8.0,17.77,17.77,533.17,2.0,379.0,0.061068,0.003053,101


,H_ch_mm,D_h_mm,u_ch_m_s,Re,Nu,h_f_W_m2K,U_W_m2K,A_required_m2,P_HT_mm,L_ch_mm,Length_above_379_mm
0,3.00,4.363636,0.127609,623.783052,5.008494,695.554664,690.750119,0.106313,14.0,379.687538,0.687538
1,3.05,4.416290,0.125517,620.960504,4.978996,683.214189,678.578047,0.108220,14.1,383.757105,4.757105
2,3.10,4.468468,0.123492,618.163385,4.949938,671.295464,666.819138,0.110128,14.2,387.774233,8.774233
3,3.15,4.520179,0.121532,615.391352,4.921314,659.778351,655.453808,0.112037,14.3,391.739368,12.739368
4,3.20,4.571429,0.119633,612.644069,4.893118,648.644010,644.463735,0.113948,14.4,395.652917,16.652917
...,...,...,...,...,...,...,...,...,...,...,...
96,7.80,7.898734,0.049080,434.279340,3.613505,277.232237,276.465784,0.265622,23.6,562.758359,183.758359
97,7.85,7.924290,0.048768,432.909374,3.598728,275.208122,274.452806,0.267570,23.7,564.493995,185.493995
98,7.90,7.949686,0.048459,431.548023,3.583132,273.140112,272.396089,0.269590,23.8,566.366454,187.366454
99,7.95,7.974922,0.048154,430.195208,3.566668,271.024706,270.292147,0.271689,23.9,568.386845,189.386845


Rectangular parallel-channel design scan
-----------------------------------------------------------------
Channel number                 : 20
Channel width                  : 8.00 mm
Channel spacing                : 17.77 mm
Edge spacing on each side      : 17.77 mm
Cold-plate width               : 533.17 mm
Plate-to-coolant distance      : 2.00 mm
Minimum channel length         : > 379.00 mm


In [6]:
# ============================================================
# Cell 6 - Pressure Drop and Pumping Power
# ============================================================

H_ch = 0.003               # Selected channel height, m
L_ch = 0.380               # Selected channel length, m

t_mission = 1920.0         # Pump operation time, s
eta_pump = 0.35            # Pump efficiency
K_minor = 0.0              # Total minor-loss coefficient

# Channel geometry
A_cool_cs = W_ch * H_ch
D_channel = 2.0 * W_ch * H_ch / (W_ch + H_ch)

# Flow conditions in one parallel channel
m_dot_ch = m_dot_f / N_ch
u_ch = m_dot_ch / (rho_f * A_cool_cs)
Re = rho_f * u_ch * D_channel / mu_f

# Darcy friction factor for a rectangular channel
f = BTMS_model.cal_friction_factor_rect_channel(Re, W_ch, H_ch)

# Pressure drop
dynamic_pressure = 0.5 * rho_f * u_ch**2
DeltaP_major = f * (L_ch / D_channel) * dynamic_pressure
DeltaP_minor = K_minor * dynamic_pressure
DeltaP = DeltaP_major + DeltaP_minor

# Total coolant volume flow rate
V_dot_f = m_dot_f / rho_f

# Hydraulic power and pump input power
P_hydraulic = DeltaP * V_dot_f
P_pump = P_hydraulic / eta_pump

# Pump energy consumption
E_pump_J = P_pump * t_mission
E_pump_Wh = E_pump_J / 3600.0

pump_summary = pd.DataFrame([{
    "Channel_number": N_ch,
    "Channel_width_mm": W_ch * 1000.0,
    "Channel_height_mm": H_ch * 1000.0,
    "Channel_length_mm": L_ch * 1000.0,
    "Hydraulic_diameter_mm": D_channel * 1000.0,
    "Mass_flow_rate_kg_s": m_dot_f,
    "Mass_flow_rate_per_channel_kg_s": m_dot_ch,
    "Channel_velocity_m_s": u_ch,
    "Re": Re,
    "Darcy_friction_factor": f,
    "Major_pressure_drop_Pa": DeltaP_major,
    "Minor_pressure_drop_Pa": DeltaP_minor,
    "Total_pressure_drop_Pa": DeltaP,
    "Volume_flow_rate_L_min": V_dot_f * 60000.0,
    "Hydraulic_power_W": P_hydraulic,
    "Pump_efficiency": eta_pump,
    "Pump_power_W": P_pump,
    "Mission_time_s": t_mission,
    "Pump_energy_J": E_pump_J,
    "Pump_energy_Wh": E_pump_Wh
}])

display(pump_summary.round(6))

print("Pressure drop and pumping power")
print("-" * 55)
print(f"Channel width          : {W_ch * 1000.0:.2f} mm")
print(f"Channel height         : {H_ch * 1000.0:.2f} mm")
print(f"Channel length         : {L_ch * 1000.0:.2f} mm")
print(f"Channel number         : {N_ch}")
print(f"Hydraulic diameter     : {D_channel * 1000.0:.4f} mm")
print(f"Channel velocity       : {u_ch:.6f} m/s")
print(f"Reynolds number        : {Re:.2f}")
print(f"Darcy friction factor  : {f:.6f}")
print(f"Major pressure drop    : {DeltaP_major:.6f} Pa")
print(f"Minor pressure drop    : {DeltaP_minor:.6f} Pa")
print(f"Total pressure drop    : {DeltaP:.6f} Pa")
print(f"Volume flow rate       : {V_dot_f * 60000.0:.6f} L/min")
print(f"Hydraulic power        : {P_hydraulic:.6f} W")
print(f"Pump efficiency        : {eta_pump:.2f}")
print(f"Pump input power       : {P_pump:.6f} W")
print(f"Pump energy            : {E_pump_Wh:.6f} Wh")

,Channel_number,Channel_width_mm,Channel_height_mm,Channel_length_mm,Hydraulic_diameter_mm,Mass_flow_rate_kg_s,Mass_flow_rate_per_channel_kg_s,Channel_velocity_m_s,Re,Darcy_friction_factor,Major_pressure_drop_Pa,Minor_pressure_drop_Pa,Total_pressure_drop_Pa,Volume_flow_rate_L_min,Hydraulic_power_W,Pump_efficiency,Pump_power_W,Mission_time_s,Pump_energy_J,Pump_energy_Wh
0,20,8.0,3.0,380.0,4.363636,0.061068,0.003053,0.127609,623.783052,0.106627,75.375264,0.0,75.375264,3.675127,0.004617,0.35,0.013191,1920.0,25.326964,0.007035


Pressure drop and pumping power
-------------------------------------------------------
Channel width          : 8.00 mm
Channel height         : 3.00 mm
Channel length         : 380.00 mm
Channel number         : 20
Hydraulic diameter     : 4.3636 mm
Channel velocity       : 0.127609 m/s
Reynolds number        : 623.78
Darcy friction factor  : 0.106627
Major pressure drop    : 75.375264 Pa
Minor pressure drop    : 0.000000 Pa
Total pressure drop    : 75.375264 Pa
Volume flow rate       : 3.675127 L/min
Hydraulic power        : 0.004617 W
Pump efficiency        : 0.35
Pump input power       : 0.013191 W
Pump energy            : 0.007035 Wh


In [7]:
# ============================================================
# Cell 7 - Liquid-Cooling BTMS Mass and Auxiliary Power
# ============================================================

rho_plate = 2719.0         # Aluminium density, kg/m3

# Total cold-plate thickness
H_plate = H_ch + delta_plate_coolant

# Pump and pipe masses are currently unavailable
m_pump = 0.0
m_pipe = 0.0

# Calculate cold-plate and coolant masses
mass_args = {
    "fluid_cool": "water",
    "rho_cool": rho_f,
    "rho_plate": rho_plate,
    "W_plate": W_plate,
    "L_channel": L_ch,
    "A_cool_cs": A_cool_cs,
    "H_channel": H_ch,
    "H_bottom": delta_plate_coolant,
    "num_channel": N_ch,
    "m_pump": m_pump,
    "m_pipe": m_pipe,
    "return_components": True,
}

mass_results = BTMS_model.cal_btms_mass(mass_args)

m_plate = mass_results["m_plate_kg"]
m_coolant = mass_results["m_coolant_kg"]
m_pump = mass_results["m_pump_kg"]
m_pipe = mass_results["m_pipe_kg"]
m_BTMS = mass_results["m_BTMS_kg"]

# ============================================================
# BTMS mass and auxiliary-power summary
# ============================================================

btms_summary = pd.DataFrame([{
    "Cold_plate_width_mm": W_plate * 1000.0,
    "Cold_plate_length_mm": L_ch * 1000.0,
    "Cold_plate_thickness_mm": H_plate * 1000.0,
    "Plate_to_coolant_distance_mm": delta_plate_coolant * 1000.0,
    "Channel_number": N_ch,
    "Channel_width_mm": W_ch * 1000.0,
    "Channel_height_mm": H_ch * 1000.0,
    "Channel_length_mm": L_ch * 1000.0,
    "Cold_plate_mass_kg": m_plate,
    "Coolant_mass_kg": m_coolant,
    "Pump_mass_kg": m_pump,
    "Pipe_mass_kg": m_pipe,
    "Total_BTMS_mass_kg": m_BTMS,
    "Pressure_drop_Pa": DeltaP,
    "Volume_flow_rate_L_min": V_dot_f * 60000.0,
    "Hydraulic_power_W": P_hydraulic,
    "Pump_efficiency": eta_pump,
    "Pump_input_power_W": P_pump,
    "Mission_time_s": t_mission,
    "Pump_energy_Wh": E_pump_Wh
}])

display(btms_summary.round(6))

print("Liquid-cooling BTMS mass and auxiliary power")
print("-" * 60)
print(f"Cold-plate width             : {W_plate * 1000.0:.2f} mm")
print(f"Cold-plate length            : {L_ch * 1000.0:.2f} mm")
print(f"Cold-plate thickness         : {H_plate * 1000.0:.2f} mm")
print(f"Plate-to-coolant distance    : {delta_plate_coolant * 1000.0:.2f} mm")
print(f"Channel number               : {N_ch}")
print(f"Channel width                : {W_ch * 1000.0:.2f} mm")
print(f"Channel height               : {H_ch * 1000.0:.2f} mm")
print(f"Channel length               : {L_ch * 1000.0:.2f} mm")
print("-" * 60)
print(f"Cold-plate mass              : {m_plate:.6f} kg")
print(f"Coolant mass                 : {m_coolant:.6f} kg")
print(f"Pump mass                    : {m_pump:.6f} kg")
print(f"Pipe mass                    : {m_pipe:.6f} kg")
print(f"Total BTMS mass              : {m_BTMS:.6f} kg")
print("-" * 60)
print(f"Pressure drop                : {DeltaP:.6f} Pa")
print(f"Volume flow rate             : {V_dot_f * 60000.0:.6f} L/min")
print(f"Hydraulic power              : {P_hydraulic:.6f} W")
print(f"Pump efficiency              : {eta_pump:.2f}")
print(f"Pump input power             : {P_pump:.6f} W")
print(f"Pump energy                  : {E_pump_Wh:.6f} Wh")

,Cold_plate_width_mm,Cold_plate_length_mm,Cold_plate_thickness_mm,Plate_to_coolant_distance_mm,Channel_number,Channel_width_mm,Channel_height_mm,Channel_length_mm,Cold_plate_mass_kg,Coolant_mass_kg,Pump_mass_kg,Pipe_mass_kg,Total_BTMS_mass_kg,Pressure_drop_Pa,Volume_flow_rate_L_min,Hydraulic_power_W,Pump_efficiency,Pump_input_power_W,Mission_time_s,Pump_energy_Wh
0,533.17,380.0,5.0,2.0,20,8.0,3.0,380.0,2.258464,0.181853,0.0,0.0,2.440317,75.375264,3.675127,0.004617,0.35,0.013191,1920.0,0.007035


Liquid-cooling BTMS mass and auxiliary power
------------------------------------------------------------
Cold-plate width             : 533.17 mm
Cold-plate length            : 380.00 mm
Cold-plate thickness         : 5.00 mm
Plate-to-coolant distance    : 2.00 mm
Channel number               : 20
Channel width                : 8.00 mm
Channel height               : 3.00 mm
Channel length               : 380.00 mm
------------------------------------------------------------
Cold-plate mass              : 2.258464 kg
Coolant mass                 : 0.181853 kg
Pump mass                    : 0.000000 kg
Pipe mass                    : 0.000000 kg
Total BTMS mass              : 2.440317 kg
------------------------------------------------------------
Pressure drop                : 75.375264 Pa
Volume flow rate             : 3.675127 L/min
Hydraulic power              : 0.004617 W
Pump efficiency              : 0.35
Pump input power             : 0.013191 W
Pump energy                  : 0.0